In [1]:
import requests
from bs4 import BeautifulSoup
import json
import re

headers = {
    "User-Agent": "YOUR_APP_OR_USER_NAME (YOUR_EMAIL_OR_CONTACT_PAGE)"
}

ancient_history_wiki_url = "https://en.wikipedia.org/api/rest_v1/page/html/Timeline_of_ancient_history"
post_classical__history_wiki_url = "https://en.wikipedia.org/api/rest_v1/page/html/Timeline_of_post-classical_history"
early_modern_history_wiki_url = "https://en.wikipedia.org/api/rest_v1/page/html/Timeline_of_geopolitical_changes_(1500%E2%80%931899)"
modern_history_wiki_url = "https://en.wikipedia.org/api/rest_v1/page/html/Timeline_of_geopolitical_changes_(1900%E2%80%931999)"

# Ancient History

In [3]:
# Fetching events from Ancient History...

In [4]:
response = requests.get(ancient_history_wiki_url, headers=headers)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

In [5]:
ids = ["mwQg", "mwAec", "mwA9U"]

data = []

sections = {i: soup.find(id=i) for i in ids}

for section_id, section in sections.items():
    if not section:
        continue

    h2 = section.find("h2")
    era = h2.get_text(strip=True) if h2 else None

    events = []

    for li in section.find_all("li"):
    
        # extract year
        b = li.find("b")
        year = b.get_text(" ", strip=True).replace(":", "") if b else None
    
        # remove subscript (citation) tags
        for sup in li.find_all("sup"):
            sup.decompose()

        # remove figure tags
        for figure in li.find_all("figure"):
            figure.decompose()
    
        if b:
            b.decompose()

        for word_see in li.find_all(string=re.compile(r"\bsee\b", re.I)):
            word_see.replace_with("-")
    
        # extract event details
        event = li.get_text(" ", strip=True)
        event = re.sub(r"\s+([.,;:!?])", r"\1", event)
        event = re.sub(r"([(\[{])\s+", r"\1", event)
        event = re.sub(r"\s+([)\]}])", r"\1", event)
    
        events.append({
            "year": year,
            "event": event
        })

    data.append({
        "era": era,
        "events": events
    })

In [6]:
#print(data)

In [7]:
# save as json
with open("./apps/backend/src/data/ancient_history.json", "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

# Post Classical History

In [9]:
# Fetching events from Post Classical History...

In [10]:
response = requests.get(post_classical__history_wiki_url, headers=headers)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

In [11]:
ids = ["mwKg", "mwBGo", "mwCBU"]

data = []

sections = {i: soup.find(id=i) for i in ids}

for section_id, section in sections.items():
    if not section:
        continue

    h2 = section.find("h2")
    era = h2.get_text(" ", strip=True).replace(":", "") if h2 else None

    era_data = {
        "era": era,
        "sections": []
    }

    for sub in section.find_all("section", recursive=False):
        h3 = sub.find("h3")
        sub_title = h3.get_text(" ", strip=True) if h3 else None

        tables = sub.find_all("table")

        events = []

        for table in tables:
            tbody = table.find("tbody")
            if not tbody:
                continue

            for tr in tbody.find_all("tr"):
                tds = tr.find_all("td")

                # skip <th>
                if len(tds) < 3:
                    continue

                # remove subscript (citation) tags
                for sup in tr.find_all("sup"):
                    sup.decompose()

                year = tds[0].get_text(" ", strip=True)
                date = tds[1].get_text(" ", strip=True)
                event = tds[2].get_text(" ", strip=True)
                event = re.sub(r"\s+([.,;:!?])", r"\1", event)
                event = re.sub(r"([(\[{])\s+", r"\1", event)
                event = re.sub(r"\s+([)\]}])", r"\1", event)
                desc = tds[3].get_text(" ", strip=True) if len(tds) > 3 else ""
                desc = re.sub(r"\s+([.,;:!?])", r"\1", desc)
                desc = re.sub(r"([(\[{])\s+", r"\1", desc)
                desc = re.sub(r"\s+([)\]}])", r"\1", desc)

                # skip empty rows
                if not year and not event:
                    continue

                events.append({
                    "year": year,
                    "date": date,
                    "event": event,
                    "description": desc
                })

        era_data["sections"].append({
            "title": sub_title,
            "events": events
        })

    data.append(era_data)

In [12]:
#print(data)

In [13]:
# save as json
with open("./apps/backend/src/data/post_classical.json", "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

# Early Modern History

In [15]:
# Fetching events from Early Modern History...

In [16]:
response = requests.get(early_modern_history_wiki_url, headers=headers)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

In [17]:
ids = ["mwFQ", "mwA8I", "mwCBg", "mwDiA"]

data = []

sections = {i: soup.find(id=i) for i in ids}

for section_id, section in sections.items():
    if not section:
        continue

    h2 = section.find("h2")
    era = h2.get_text(" ", strip=True).replace(":", "") if h2 else None

    era_data = {
        "era": era,
        "sections": []
    }

    # for the first section that doesn't have sub sections
    first_tables = section.find_all("table", recursive=False)

    if first_tables:
        events = []
        current_year = None

        for table in first_tables:
            tbody = table.find("tbody")
            if not tbody:
                continue

            for tr in tbody.find_all("tr"):
                tds = tr.find_all("td")

                if not tds:
                    continue

                # remove subscript (citation) tags
                for sup in tr.find_all("sup"):
                    sup.decompose()

                texts = [td.get_text(" ", strip=True) for td in tds]

                if len(texts) >= 3:
                    current_year = texts[0]
                    event = texts[-1]

                elif len(texts) == 2:
                    event = texts[1]

                elif len(texts) == 1:
                    event = texts[0]

                else:
                    continue

                if not current_year or not event:
                    continue

                
                event = re.sub(r"\s+([.,;:!?])", r"\1", event)
                event = re.sub(r"([(\[{])\s+", r"\1", event)
                event = re.sub(r"\s+([)\]}])", r"\1", event)
                
                events.append({
                    "year": current_year,
                    "event": event
                })

        era_data["sections"].append({
            "title": "1500",
            "events": events
        })

    # for sections that have sub sections...
    for h3 in section.find_all("h3"):
        sub_title = h3.get_text(" ", strip=True)

        table = h3.find_next("table")
        if not table:
            continue

        tbody = table.find("tbody")
        if not tbody:
            continue

        events = []
        current_year = None

        for tr in tbody.find_all("tr"):
            tds = tr.find_all("td")

            if not tds:
                continue

            # remove subscript (citation) tags
            for sup in tr.find_all("sup"):
                sup.decompose()

            texts = [td.get_text(" ", strip=True) for td in tds]

            if len(texts) >= 3:
                current_year = texts[0]
                event = texts[-1]

            elif len(texts) == 2:
                event = texts[1]

            elif len(texts) == 1:
                event = texts[0]

            else:
                continue

            if not current_year or not event:
                continue

            
            event = re.sub(r"\s+([.,;:!?])", r"\1", event)
            event = re.sub(r"([(\[{])\s+", r"\1", event)
            event = re.sub(r"\s+([)\]}])", r"\1", event)
            
            events.append({
                "year": current_year,
                "event": event
            })

        era_data["sections"].append({
            "title": sub_title,
            "events": events
        })

    data.append(era_data)

In [18]:
#print(data)

In [19]:
# save as json
with open("./apps/backend/src/data/early_modern.json", "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)

# Modern History

In [21]:
# Fetching events from Late Modern History...

In [22]:
response = requests.get(modern_history_wiki_url, headers=headers)
response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")

In [23]:
ids = ['mwNA','mwAkc','mwBlo','mwCNM','mwDHs','mwGA4','mwHJw','mwIhI','mwKU4','mwLKo']

data = []

sections = {i: soup.find(id=i) for i in ids}

for section_id, section in sections.items():
    if not section:
        continue

    h2 = section.find("h2")
    era = h2.get_text(strip=True) if h2 else None

    first_tables = section.find_all("table", recursive=False)

    if first_tables:
        events = []
        current_year = None

        for table in first_tables:
            tbody = table.find("tbody")
            if not tbody:
                continue

            for tr in tbody.find_all("tr"):
                tds = tr.find_all("td")

                if not tds:
                    continue

                # remove subscript (citation) tags
                for sup in tr.find_all("sup"):
                    sup.decompose()

                texts = [td.get_text(" ", strip=True) for td in tds]

                if len(texts) >= 3:
                    current_year = texts[0]
                    event = texts[-1]

                elif len(texts) == 2:
                    event = texts[1]

                elif len(texts) == 1:
                    event = texts[0]

                else:
                    continue

                if not current_year or not event:
                    continue

                event = re.sub(r"\s+([.,;:!?])", r"\1", event)
                event = re.sub(r"([(\[{])\s+", r"\1", event)
                event = re.sub(r"\s+([)\]}])", r"\1", event)

                events.append({
                    "year": current_year,
                    "event": event
                })


    data.append({
        "era": era,
        "events": events
    })

In [24]:
#print(data)

In [25]:
# save as json
with open("./apps/backend/src/data/late_modern.json", "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)